[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/31_transformers/04_encoder_decoder_bert_gpt_vit.ipynb)

# Bài 04 — Encoder–decoder, BERT, GPT, ViT

**🎯 Mục tiêu**
- Ba họ kiến trúc: **encoder-only** (BERT), **decoder-only** (GPT), **encoder–decoder** (Transformer gốc, T5) — mask và bài toán phù hợp.
- **Encoder–decoder** với cross-attention trên bài toán đảo chuỗi của chương 30.
- **BERT**: masked language modeling (MLM), tiền huấn luyện rồi fine-tune khi ít nhãn — và so với baseline đơn giản.
- **ViT**: ảnh → các patch → Transformer; so với CNN theo lượng dữ liệu (inductive bias).
- Dùng mô hình tiền huấn luyện trong thực tế (Hugging Face) và khi nào **không** cần Transformer.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/31_transformers

In [ ]:
import copy
import math
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.datasets import SHAPE_CLASSES, make_shapes
from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. Ba họ kiến trúc

| | Encoder-only | Decoder-only | Encoder–decoder |
|---|---|---|---|
| Ví dụ | BERT, RoBERTa, PhoBERT | GPT, LLaMA, Claude | Transformer gốc, T5, BART, Whisper |
| Attention | hai chiều (không causal mask) | causal | encoder hai chiều; decoder causal + **cross-attention** tới encoder |
| Mục tiêu tiền huấn luyện | điền token bị che (MLM) | dự đoán token tiếp theo | khôi phục đoạn bị che / dịch |
| Mạnh ở | phân loại, gán nhãn token, embedding/tìm kiếm | sinh văn bản, học trong ngữ cảnh, hầu hết tác vụ khi đủ lớn | chuỗi → chuỗi: dịch, tóm tắt, nhận dạng tiếng nói |

Cả ba dùng cùng một khối (bài 02); khác nhau ở **mask** và **mục tiêu huấn luyện**. ViT (mục 4) là encoder-only áp dụng cho ảnh.

## 2. Encoder–decoder: cross-attention

Decoder có thêm một lớp **cross-attention** trong mỗi khối: **query** từ decoder, **key/value** từ đầu ra encoder ("memory"). Đây chính là attention seq2seq của chương 30, nhưng cả encoder và decoder đều là chồng khối Transformer thay vì RNN.

Bài toán: **đảo ngược dãy chữ số** độ dài 3–15 (giống chương 30, cùng token `PAD`, `SOS`, `EOS`). Kết quả chương 30 (GRU + attention, 2 000 bước): khớp cả chuỗi ≈ 1.0 / 1.0 / 0.99 ở độ dài 5 / 10 / 15 và 0 ở độ dài 20 (chưa thấy khi train).

⚠️ Quy ước mask của `nn.Transformer*` **ngược** với hàm tự viết ở bài 01–03: mask bool `True` = **bị che**. Nhầm quy ước này là lỗi rất phổ biến.

In [ ]:
PAD, SOS, EOS, V = 10, 11, 12, 13


def make_batch(n, g, min_len=3, max_len=15):
    lens = torch.randint(min_len, max_len + 1, (n,), generator=g)
    L = int(lens.max())
    src = torch.full((n, L), PAD)
    tgt_in = torch.full((n, L + 1), PAD)
    tgt_out = torch.full((n, L + 1), PAD)
    for i, l in enumerate(lens.tolist()):
        seq = torch.randint(0, 10, (l,), generator=g)
        src[i, :l] = seq
        tgt_in[i, 0] = SOS
        tgt_in[i, 1:l + 1] = seq.flip(0)
        tgt_out[i, :l] = seq.flip(0)
        tgt_out[i, l] = EOS
    return src, lens, tgt_in, tgt_out


def sinusoidal_encoding(T, d):
    pos = torch.arange(T)[:, None]
    freq = 1.0 / 10000 ** (torch.arange(0, d, 2) / d)
    pe = torch.zeros(T, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * freq), torch.cos(pos * freq)
    return pe


def future_mask(T):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)   # True = bị che (quy ước PyTorch)


class TransformerSeq2Seq(nn.Module):
    def __init__(self, d=64, n_heads=4, n_layers=2, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=PAD)
        self.register_buffer("pos", sinusoidal_encoding(max_len, d))
        enc_layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        dec_layer = nn.TransformerDecoderLayer(d, n_heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers, norm=nn.LayerNorm(d), enable_nested_tensor=False)
        self.decoder = nn.TransformerDecoder(dec_layer, n_layers, norm=nn.LayerNorm(d))
        self.out = nn.Linear(d, V)

    def embed(self, x):
        return self.emb(x) + self.pos[: x.shape[1]]

    def encode(self, src):
        return self.encoder(self.embed(src), src_key_padding_mask=src == PAD)

    def decode(self, tgt_in, memory, src):
        h = self.decoder(self.embed(tgt_in), memory, tgt_mask=future_mask(tgt_in.shape[1]),
                         tgt_key_padding_mask=tgt_in == PAD, memory_key_padding_mask=src == PAD)
        return self.out(h)

    def forward(self, src, tgt_in):
        return self.decode(tgt_in, self.encode(src), src)

    @torch.no_grad()
    def greedy(self, src, max_len):
        memory = self.encode(src)
        ys = torch.full((src.shape[0], 1), SOS, device=src.device)
        for _ in range(max_len):
            ys = torch.cat([ys, self.decode(ys, memory, src)[:, -1].argmax(-1, keepdim=True)], dim=1)
        return ys[:, 1:]


def train_seq2seq(steps=3000, warmup=200, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = TransformerSeq2Seq().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * s / steps)))
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        model.train()
        src, _, tgt_in, tgt_out = make_batch(128, g)
        opt.zero_grad()
        logits = model(src.to(device), tgt_in.to(device))
        nn.functional.cross_entropy(logits.reshape(-1, V), tgt_out.to(device).reshape(-1), ignore_index=PAD).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
    return model.eval()


def exact_match(model, length, n=500):
    src, _, _, tgt_out = make_batch(n, torch.Generator().manual_seed(1000 + length), length, length)
    return (model.greedy(src.to(device), length + 1).cpu() == tgt_out).all(1).float().mean().item()


t0 = time.perf_counter()
s2s = train_seq2seq()
print(f"Transformer encoder–decoder: {sum(p.numel() for p in s2s.parameters()):,} tham số | {time.perf_counter() - t0:.0f}s")
for L in [5, 10, 15, 20]:
    print(f"  độ dài {L:2d}: khớp cả chuỗi = {exact_match(s2s, L):.3f}" + ("  ← chưa thấy khi train" if L > 15 else ""))

Trong phạm vi độ dài đã train, encoder–decoder Transformer giải hoàn toàn bài toán, ngang GRU + attention; ở độ dài 20 cả hai đều thất bại — mã hoá vị trí sinusoidal **về lý thuyết** tính được cho mọi vị trí, nhưng mô hình chưa từng thấy các vị trí đó khi train nên không dùng đúng được. Ngoại suy độ dài là vấn đề mở (RoPE + kỹ thuật nội suy, ALiBi giúp phần nào).

Một chi tiết cài đặt đáng nhớ: bài báo gốc nhân embedding với $\sqrt{d}$ trước khi cộng mã hoá vị trí — hợp lý khi embedding khởi tạo nhỏ (và dùng chung với lớp đầu ra). Với `nn.Embedding` khởi tạo $\mathcal N(0, 1)$ mặc định, nhân thêm $\sqrt{d}$ khiến embedding lớn gấp nhiều lần mã hoá vị trí, "át" thông tin vị trí — khi soạn bài, thay đổi nhỏ này làm độ chính xác ở chuỗi dài giảm rõ rệt. Bài học: tỉ lệ giữa các thành phần được **cộng** vào residual stream rất quan trọng.

## 3. BERT: masked language modeling và fine-tuning

BERT (Devlin et al., 2019) là encoder hai chiều, tiền huấn luyện trên văn bản **không nhãn** bằng **MLM**: chọn ngẫu nhiên 15% token; trong đó 80% thay bằng `[MASK]`, 10% thay bằng token ngẫu nhiên, 10% giữ nguyên; mô hình dự đoán token gốc **tại các vị trí được chọn**. (10% + 10% để mô hình không chỉ học "khi thấy `[MASK]` mới cần dự đoán", vì lúc fine-tune không có `[MASK]`.) Token `[CLS]` đầu câu: biểu diễn cuối của nó dùng cho phân loại.

Thí nghiệm thu nhỏ trên bộ review: tách từ đơn giản, từ vựng từ tập train.

In [ ]:
reviews = pd.read_csv("../../data/sample/reviews_vi.csv")
tokenize = lambda s: re.findall(r"\w+|[^\w\s]", s.lower())
rv_train, rv_test = train_test_split(reviews, test_size=0.3, random_state=0, stratify=reviews.sentiment)

PAD_ID, UNK_ID, CLS_ID, MASK_ID = 0, 1, 2, 3
words = sorted({w for s in rv_train.text for w in tokenize(s)})
vocab = {w: i + 4 for i, w in enumerate(words)}
itos = ["[PAD]", "[UNK]", "[CLS]", "[MASK]"] + words
VOCAB = len(itos)
LABELS = {"neg": 0, "neu": 1, "pos": 2}


def encode_texts(texts, max_len=40):
    X = torch.full((len(texts), max_len), PAD_ID)
    for i, s in enumerate(texts):
        ids = [CLS_ID] + [vocab.get(w, UNK_ID) for w in tokenize(s)][: max_len - 1]
        X[i, :len(ids)] = torch.tensor(ids)
    return X


X_rv_train, X_rv_test = encode_texts(rv_train.text), encode_texts(rv_test.text)
y_rv_train = torch.tensor(rv_train.sentiment.map(LABELS).values)
y_rv_test = torch.tensor(rv_test.sentiment.map(LABELS).values)
print(f"từ vựng: {VOCAB} token | train {len(X_rv_train)} câu | test {len(X_rv_test)} câu")


def mask_tokens(x, g, p=0.15):
    x = x.clone()
    labels = torch.full_like(x, -100)                                  # -100: cross_entropy bỏ qua
    candidates = (x != PAD_ID) & (x != CLS_ID)
    chosen = (torch.rand(x.shape, generator=g) < p) & candidates
    labels[chosen] = x[chosen]
    r = torch.rand(x.shape, generator=g)
    x[chosen & (r < 0.8)] = MASK_ID                                    # 80%: [MASK]
    random_tok = chosen & (r >= 0.8) & (r < 0.9)                       # 10%: token ngẫu nhiên; 10% còn lại giữ nguyên
    x[random_tok] = torch.randint(4, VOCAB, (int(random_tok.sum()),), generator=g)
    return x, labels


xm, lab = mask_tokens(X_rv_train[:3], torch.Generator().manual_seed(1), p=0.3)
for row, l in zip(xm, lab):
    print(" ".join(itos[t] for t in row.tolist() if t != PAD_ID), "| cần đoán:", [itos[t] for t in l.tolist() if t != -100])

In [ ]:
class MiniBERT(nn.Module):
    def __init__(self, d=64, n_heads=4, n_layers=2, max_len=40, n_classes=3):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d, padding_idx=PAD_ID)
        self.pos = nn.Parameter(torch.randn(max_len, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.1, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d)
        self.mlm_head = nn.Linear(d, VOCAB)                            # dùng khi tiền huấn luyện
        self.cls_head = nn.Linear(d, n_classes)                        # dùng khi fine-tune

    def hidden(self, x):
        return self.ln(self.encoder(self.emb(x) + self.pos[: x.shape[1]], src_key_padding_mask=x == PAD_ID))

    def forward(self, x):
        return self.cls_head(self.hidden(x)[:, 0])                     # biểu diễn của [CLS]


def pretrain_mlm(steps=3000, seed=0):
    torch.manual_seed(seed)
    model = MiniBERT().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        model.train()
        idx = torch.randint(0, len(X_rv_train), (128,), generator=g)
        x, labels = mask_tokens(X_rv_train[idx], g)                    # chỉ dùng văn bản, KHÔNG dùng nhãn
        opt.zero_grad()
        logits = model.mlm_head(model.hidden(x.to(device)))
        nn.functional.cross_entropy(logits.reshape(-1, VOCAB), labels.to(device).reshape(-1)).backward()
        opt.step()
    return model.eval()


t0 = time.perf_counter()
bert = pretrain_mlm()
print(f"tiền huấn luyện MLM: {time.perf_counter() - t0:.0f}s")


@torch.no_grad()
def fill_mask(sentence, position):
    ids = [CLS_ID] + [vocab[w] for w in tokenize(sentence)]
    target = itos[ids[position]]
    ids[position] = MASK_ID
    probs = bert.mlm_head(bert.hidden(torch.tensor([ids], device=device)))[0, position].softmax(-1)
    top = probs.topk(3)
    masked = " ".join(itos[t] for t in ids[1:])
    print(f"{masked:32s} (gốc: {target:6s}) → " + ", ".join(f"{itos[i]} {p:.2f}" for p, i in zip(top.values.tolist(), top.indices.tolist())))


for sentence, position in [("pin rất yếu", 3), ("chất lượng kém", 3), ("giao hàng nhanh", 3), ("đóng gói cẩn thận", 4)]:
    fill_mask(sentence, position)

Không ai cho mô hình nhãn, nhưng để điền đúng từ bị che, nó phải học cách các từ đi với nhau trong ngữ liệu (collocation, cấu trúc câu review). Biểu diễn học được có dùng cho tác vụ có nhãn không? So sánh khi chỉ có **30** hoặc **100** câu có nhãn (3 lần chọn mẫu khác nhau):
- **Từ đầu**: cùng kiến trúc, khởi tạo ngẫu nhiên.
- **Fine-tune**: khởi tạo từ mô hình MLM, train **toàn bộ** với nhãn.
- **TF-IDF + logistic regression** (chương 32) — baseline cổ điển.

In [ ]:
def finetune(model, idx, epochs, lr=5e-4, seed=0):
    torch.manual_seed(seed)
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    X, y = X_rv_train[idx].to(device), y_rv_train[idx].to(device)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X))
        for s in range(0, len(X), 16):
            b = perm[s:s + 16]
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[b]), y[b]).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return (model(X_rv_test.to(device)).argmax(1).cpu() == y_rv_test).float().mean().item()


def tfidf_baseline(idx):
    texts = rv_train.text.values[idx.numpy()]
    vec = TfidfVectorizer(ngram_range=(1, 2)).fit(texts)
    clf = LogisticRegression(max_iter=2000, C=10).fit(vec.transform(texts), y_rv_train[idx].numpy())
    return (clf.predict(vec.transform(rv_test.text)) == y_rv_test.numpy()).mean()


for n_labels in [30, 100]:
    res = {"từ đầu": [], "fine-tune từ MLM": [], "TF-IDF + LogReg": []}
    epochs = 60 * 30 // n_labels                                       # cùng số bước cập nhật cho mọi n_labels
    for seed in range(3):
        idx = torch.tensor(np.random.default_rng(seed).choice(len(X_rv_train), n_labels, replace=False))
        res["từ đầu"].append(finetune(MiniBERT(), idx, epochs, seed=seed))
        res["fine-tune từ MLM"].append(finetune(copy.deepcopy(bert), idx, epochs, seed=seed))
        res["TF-IDF + LogReg"].append(tfidf_baseline(idx))
    print(f"{n_labels:3d} nhãn: " + " | ".join(f"{k} = {np.mean(v):.3f} ± {np.std(v):.3f}" for k, v in res.items()))

- Fine-tune từ MLM **tốt hơn** train từ đầu ở cùng lượng nhãn: biểu diễn học từ văn bản không nhãn được tái sử dụng — đó là toàn bộ ý tưởng của "pretrain → fine-tune". Với 30 nhãn, độ lệch giữa các lần chọn mẫu lớn (lệch chuẩn ~0.07), nên chênh lệch ở mức đó chưa chắc chắn; với 100 nhãn chênh lệch rõ và ổn định.
- Nhưng **TF-IDF + logistic regression ngang bằng** mô hình fine-tune: bộ review này đơn giản (từ vựng < 100 từ, cảm xúc nằm ở vài từ khoá). Lợi ích thật của tiền huấn luyện xuất hiện khi ngôn ngữ phức tạp (phủ định, mỉa mai, ngữ cảnh dài) và mô hình được tiền huấn luyện trên **lượng văn bản khổng lồ** — vd PhoBERT trên 20GB văn bản tiếng Việt. Luôn so với baseline đơn giản trước khi dùng Transformer.

## 4. ViT: ảnh là một chuỗi patch

Vision Transformer (Dosovitskiy et al., 2021): cắt ảnh thành các **patch** $P\times P$ không chồng lấn, làm phẳng và chiếu tuyến tính mỗi patch thành một token (tương đương Conv có kernel = stride = $P$), thêm token `[CLS]` và position embedding học được, rồi đưa qua encoder Transformer. Ảnh 28×28 với $P = 7$ → 16 token.

In [ ]:
X_img, y_img = make_shapes(30000, seed=0)
X_img_test, y_img_test = make_shapes(1000, seed=1)
to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)
X_img, X_img_test = to_tensor(X_img), to_tensor(X_img_test)
y_img, y_img_test = torch.tensor(y_img), torch.tensor(y_img_test)

patches = X_img[0, 0].unfold(0, 7, 7).unfold(1, 7, 7)                  # (4, 4, 7, 7)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), gridspec_kw={"width_ratios": [1, 1.4]})
axes[0].imshow(X_img[0, 0], cmap="gray")
axes[0].set_title(f"ảnh gốc ({SHAPE_CLASSES[y_img[0]]})")
grid = torch.ones(4 * 8 - 1, 4 * 8 - 1)
for i in range(4):
    for j in range(4):
        grid[i * 8:i * 8 + 7, j * 8:j * 8 + 7] = patches[i, j]
axes[1].imshow(grid, cmap="gray")
axes[1].set_title("16 patch 7×7 = 16 token")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class ViT(nn.Module):
    def __init__(self, img=28, patch=7, d=64, n_heads=4, n_layers=4, n_classes=5):
        super().__init__()
        n_patches = (img // patch) ** 2
        self.patch_embed = nn.Conv2d(1, d, kernel_size=patch, stride=patch)   # = chiếu tuyến tính từng patch
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n_patches + 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)

    def forward(self, x):
        tokens = self.patch_embed(x).flatten(2).transpose(1, 2)        # (B, 16, d)
        h = torch.cat([self.cls.expand(len(x), -1, -1), tokens], dim=1) + self.pos
        return self.head(self.ln(self.encoder(h)[:, 0]))


def make_cnn():                                                        # "CNN + flatten" của chương 29
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(64 * 7 * 7, 5))


def train_image_model(model, X, y, steps=4000, seed=0):
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    done = 0
    while done < steps:
        perm = torch.randperm(len(X), generator=g)
        for s in range(0, len(perm) - 63, 64):
            model.train()
            idx = perm[s:s + 64]
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx].to(device)), y[idx].to(device)).backward()
            opt.step()
            done += 1
            if done == steps:
                break
    model.eval()
    with torch.no_grad():
        return (model(X_img_test.to(device)).argmax(1).cpu() == y_img_test).float().mean().item()


for n_train in [3000, 30000]:
    for name, make in [("CNN", make_cnn), ("ViT patch 7", ViT)]:
        torch.manual_seed(0)
        model = make()
        t0 = time.perf_counter()
        acc = train_image_model(model, X_img[:n_train], y_img[:n_train])
        print(f"{n_train:6,d} ảnh train | {name:11s}: {sum(p.numel() for p in model.parameters()):7,d} tham số | "
              f"accuracy test = {acc:.3f} | {time.perf_counter() - t0:.0f}s")

Cùng 4 000 bước cập nhật:
- Với 3 000 ảnh, CNN vượt xa ViT dù ViT có nhiều tham số hơn ~5 lần. Với 30 000 ảnh, khoảng cách thu hẹp đáng kể nhưng CNN vẫn dẫn.
- Lý do: **inductive bias**. CNN "biết sẵn" tính cục bộ và bất biến dịch (chương 29); ViT phải **học** quan hệ giữa các patch từ dữ liệu — linh hoạt hơn nhưng cần nhiều dữ liệu hơn. Bài báo ViT gốc chỉ vượt CNN khi tiền huấn luyện trên hàng trăm triệu ảnh; DeiT đạt kết quả tốt trên ImageNet nhờ augmentation mạnh và chưng cất (distillation).
- Thực tế: với ít ảnh, dùng CNN hoặc **fine-tune** một ViT/CNN đã tiền huấn luyện (chương 33).

## 5. Dùng mô hình tiền huấn luyện trong thực tế

Không ai train BERT/GPT từ đầu cho một bài toán nhỏ: tải mô hình tiền huấn luyện (Hugging Face Hub) rồi fine-tune hoặc dùng trực tiếp. Đoạn code dưới đây là **tham khảo** — cần `pip install transformers` và tải mô hình (vài trăm MB) nên mặc định không chạy (`RUN_HF = False`).

In [ ]:
RUN_HF = False                                                         # đổi thành True nếu đã cài transformers và có mạng

if RUN_HF:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

    fill = pipeline("fill-mask", model="xlm-roberta-base")             # encoder đa ngôn ngữ, có tiếng Việt
    for r in fill("Giao hàng rất <mask>, tôi rất hài lòng.")[:3]:
        print(r["token_str"], round(r["score"], 3))

    # Fine-tune cho phân loại cảm xúc: thêm đầu phân loại lên encoder tiền huấn luyện
    name = "xlm-roberta-base"                                          # PhoBERT (vinai/phobert-base) tốt hơn cho tiếng Việt nhưng cần tách từ trước
    tok = AutoTokenizer.from_pretrained(name)
    clf = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
    batch = tok(list(rv_train.text[:8]), padding=True, truncation=True, return_tensors="pt")
    out = clf(**batch, labels=y_rv_train[:8])
    print("loss một batch:", out.loss.item(), "| logits:", tuple(out.logits.shape))
    # → vòng lặp train như các bài trước (AdamW, lr ~2e-5, 2–4 epoch), hoặc dùng transformers.Trainer
else:
    print("Bỏ qua phần Hugging Face (RUN_HF = False).")

## 6. Chọn kiến trúc nào?

| Bài toán | Lựa chọn hợp lý |
|---|---|
| Dữ liệu bảng | Gradient boosting (chương 13–14) — Transformer hiếm khi thắng |
| Phân loại văn bản, ít dữ liệu | Baseline TF-IDF + tuyến tính → fine-tune encoder tiền huấn luyện (PhoBERT...) |
| Tìm kiếm ngữ nghĩa, embedding | Encoder (sentence embedding, chương 32) |
| Sinh văn bản, trợ lý, nhiều tác vụ | Decoder-only lớn (dùng API hoặc mô hình mở) + prompt / fine-tune |
| Dịch, tóm tắt, tiếng nói → văn bản | Encoder–decoder (T5, BART, Whisper) hoặc decoder-only lớn |
| Ảnh, ít dữ liệu | CNN hoặc fine-tune mô hình tiền huấn luyện (CNN/ViT) — chương 33 |
| Chuỗi rất dài, tài nguyên hạn chế | Attention hiệu quả (FlashAttention, cửa sổ trượt), SSM, hoặc mô hình nhỏ hơn |

## 🧠 Kiểm tra nhanh

1. Encoder-only, decoder-only, encoder–decoder khác nhau về mask và mục tiêu huấn luyện thế nào?
2. Cross-attention lấy query, key, value từ đâu?
3. Mô tả MLM. Vì sao không thay 100% token được chọn bằng `[MASK]`?
4. Vì sao fine-tune từ mô hình tiền huấn luyện tốt hơn train từ đầu khi ít nhãn? Thí nghiệm cho thấy điều gì về baseline?
5. ViT biến ảnh thành chuỗi token thế nào? Vì sao ViT cần nhiều dữ liệu hơn CNN?

## 📌 Tóm tắt

- Cùng khối Transformer, ba họ khác nhau ở mask và mục tiêu: BERT (hai chiều, MLM), GPT (causal, token tiếp theo), encoder–decoder (cross-attention).
- Encoder–decoder Transformer giải bài đảo chuỗi trong phạm vi độ dài đã train; vẫn không ngoại suy độ dài.
- Pretrain (không nhãn) → fine-tune (ít nhãn) tốt hơn train từ đầu; nhưng phải so với baseline — trên dữ liệu dễ, TF-IDF + LogReg ngang bằng.
- ViT: patch → token; ít inductive bias → cần nhiều dữ liệu hoặc tiền huấn luyện; với ít ảnh, CNN thắng.

## 📚 Tài liệu tham khảo

- Devlin et al. (2019) — *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*
- Raffel et al. (2020) — *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer* (T5)
- Dosovitskiy et al. (2021) — *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale* (ViT)
- Touvron et al. (2021) — *Training data-efficient image transformers & distillation through attention* (DeiT)
- Nguyen & Nguyen (2020) — *PhoBERT: Pre-trained language models for Vietnamese*
- [Hugging Face Course](https://huggingface.co/learn/nlp-course)